# Diagnóstico da Demanda Potencial por Assistência Estudantil — Campus I UFPB

## Alinhamento terminológico

> Este notebook substitui a versão anterior, que rotulava `IDPNA` como "Índice de Desempenho Acadêmico" — um erro de nomenclatura, já que a base não contém nenhuma variável de desempenho. Conforme a documentação formal do projeto (*Análise do Perfil Socioeconômico e da Demanda por Assistência Estudantil*), o indicador correto é o **IDPNA — Índice de Demanda Potencial Não Atendida**, equivalente conceitual ao "Indicador de Demanda Reprimida (IDR)" descrito na metodologia do trabalho.
>
> **Definição implementada** (`src/facts/fato_assistencia.py`):
> ```python
> IDPNA = 1  se  IN_RESERVA_VAGAS == 1  E  IN_APOIO_SOCIAL == 0
> IDPNA = 0  caso contrário
> ```
> Ou seja: **estudante que ingressou por reserva de vagas (cotas) e ainda não recebe apoio social** — a população que atende ao critério de vulnerabilidade (cota) mas não tem cobertura assistencial efetiva.
>
> Diferença em relação ao desenho original do documento: em vez de agregar `QT_MAT_RESERVA_VAGA` e `QT_MAT_APOIO_SOCIAL` diretamente dos microdados brutos do INEP por curso, o pipeline atual calcula o indicador no nível de grupo (combinação curso/centro/sexo/raça/turno/auxílio) a partir da extração do SEDAP+, com `TOTAL_ALUNOS` como peso. O resultado agregado por curso é equivalente ao `IDR_C` proposto, mas permite também quebras demográficas (Pergunta 4) que a agregação simples por curso não permitiria.

## Perguntas analíticas deste notebook

1. Qual o percentual de cotistas do Campus I que efetivamente contam com apoio social?
2. Cursos do turno noturno apresentam taxa de desassistência proporcionalmente maior que os cursos diurnos?
3. Quais os 10 cursos com maior volume absoluto de estudantes em situação de demanda não atendida (IDPNA)?
4. Como o perfil demográfico (raça/cor e sexo) dos estudantes assistidos se compara ao do corpo discente total?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from scipy.stats import chi2_contingency
except ImportError as e:
    raise ImportError(
        "Este notebook usa scipy para o teste Qui-Quadrado (Pergunta 2). "
        "Instale com: pip install scipy"
    ) from e

pd.set_option("display.float_format", lambda x: "%.2f" % x)
sns.set_theme(style="whitegrid")

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PATH_FATO = BASE_DIR / "data" / "processed" / "Fato" / "fato_assistencia.csv"
PATH_DIM_RACA = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_raca.csv"
PATH_DIM_TURNO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_turno.csv"
PATH_DIM_SEXO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_sexo.csv"
PATH_DIM_CURSO = (
    BASE_DIR / "data" / "processed" / "Dimensões" / "dim_curso_ufpb_campus_1.csv"
)

df_fato = pd.read_csv(PATH_FATO, sep=";")

df_raca = pd.read_csv(PATH_DIM_RACA, sep=";")
df_fato = df_fato.merge(df_raca[["ID_RACA", "DESCRICAO"]], on="ID_RACA", how="left")
df_fato.rename(columns={"DESCRICAO": "RACA_DESCRICAO"}, inplace=True)

df_turno = pd.read_csv(PATH_DIM_TURNO, sep=";")
df_fato = df_fato.merge(df_turno[["ID_TURNO", "DESCRICAO"]], on="ID_TURNO", how="left")
df_fato.rename(columns={"DESCRICAO": "TURNO_DESCRICAO"}, inplace=True)

df_sexo = pd.read_csv(PATH_DIM_SEXO, sep=";")
df_fato = df_fato.merge(df_sexo[["ID_SEXO", "DESCRICAO"]], on="ID_SEXO", how="left")
df_fato.rename(columns={"DESCRICAO": "SEXO_DESCRICAO"}, inplace=True)

df_curso = pd.read_csv(PATH_DIM_CURSO, sep=";")
df_fato = df_fato.merge(
    df_curso[["CO_CURSO", "NO_CURSO"]], on="CO_CURSO", how="left"
)
df_fato.rename(columns={"NO_CURSO": "CURSO"}, inplace=True)

print(f"Base carregada: {len(df_fato):,} registros agrupados")
print(f"Total de matrículas representadas (TOTAL_ALUNOS): {df_fato['TOTAL_ALUNOS'].sum():,}")
print(f"Cursos sem NO_CURSO mapeado: {df_fato['CURSO'].isna().sum()} registros")

## Pergunta 1 — Qual o percentual de cotistas que efetivamente contam com apoio social?

População de referência: estudantes com `IN_RESERVA_VAGAS = 1`. Dentro desse grupo, `RECEBE_AUXILIO = 1` indica cobertura efetiva; `IDPNA = 1` (equivalente a `RECEBE_AUXILIO = 0` neste subconjunto) indica demanda não atendida.

In [ ]:
df_cotistas = df_fato[df_fato["IN_RESERVA_VAGAS"] == 1].copy()

total_cotistas = df_cotistas["TOTAL_ALUNOS"].sum()
cotistas_com_apoio = df_cotistas.loc[
    df_cotistas["RECEBE_AUXILIO"] == 1, "TOTAL_ALUNOS"
].sum()
cotistas_sem_apoio = total_cotistas - cotistas_com_apoio

pct_com_apoio = cotistas_com_apoio / total_cotistas * 100
pct_sem_apoio = cotistas_sem_apoio / total_cotistas * 100

resumo_q1 = pd.DataFrame(
    {
        "Situação": ["Cotistas com apoio social", "Cotistas sem apoio social (IDPNA)"],
        "Total de Alunos": [cotistas_com_apoio, cotistas_sem_apoio],
        "% do total de cotistas": [round(pct_com_apoio, 2), round(pct_sem_apoio, 2)],
    }
)

print("=" * 60)
print("RESPOSTA — PERGUNTA 1")
print("=" * 60)
print(f"Total de cotistas no Campus I: {total_cotistas:,.0f}")
print(f"Cobertura efetiva de apoio social entre cotistas: {pct_com_apoio:.2f}%")
print(f"Demanda potencial não atendida entre cotistas: {pct_sem_apoio:.2f}%")
display(resumo_q1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    resumo_q1["Total de Alunos"],
    labels=resumo_q1["Situação"],
    autopct="%1.1f%%",
    colors=["#3182bd", "#e6550d"],
    startangle=90,
)
ax.set_title("Cobertura de Apoio Social entre Cotistas — Campus I", fontweight="bold")
plt.tight_layout()
plt.show()

## Pergunta 2 — Cursos do turno noturno têm taxa de desassistência maior que os diurnos?

Agrupamos os turnos em dois períodos (Matutino, Vespertino e Integral → **Diurno**; Noturno → **Noturno**; registros "Não informado" são excluídos do teste por não indicarem período real). A comparação é feita **dentro da população de cotistas** (mesma população da Pergunta 1), medindo a proporção de `IDPNA = 1` em cada período.

Teste de hipótese: **Qui-Quadrado de Independência**, avaliando se a proporção de desassistência é estatisticamente independente do período do curso.
- H0: a taxa de desassistência é independente do período (diurno/noturno).
- H1: há associação entre período e taxa de desassistência.

O documento do projeto especifica α = 0,5%. Como esse valor é incomum (o padrão em ciências sociais aplicadas é 5%), reportamos o p-valor bruto e avaliamos contra os dois limiares (0,05 e 0,005) para que a conclusão não dependa de uma possível imprecisão de digitação no texto original.

In [ ]:
def bucket_periodo(turno_desc):
    if turno_desc == "Noturno":
        return "Noturno"
    if turno_desc == "Não informado":
        return np.nan
    return "Diurno"


df_cotistas["PERIODO"] = df_cotistas["TURNO_DESCRICAO"].apply(bucket_periodo)
df_cotistas_periodo = df_cotistas.dropna(subset=["PERIODO"]).copy()

excluidos = len(df_cotistas) - len(df_cotistas_periodo)
print(f"Registros de cotistas com turno 'Não informado' excluídos do teste: {excluidos}")

# Tabela de contingência ponderada por TOTAL_ALUNOS
df_cotistas_periodo["STATUS"] = np.where(
    df_cotistas_periodo["IDPNA"] == 1, "Desassistido", "Assistido"
)

tabela_contingencia = (
    df_cotistas_periodo.groupby(["PERIODO", "STATUS"])["TOTAL_ALUNOS"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=["Assistido", "Desassistido"])
)

tabela_contingencia["Total"] = tabela_contingencia.sum(axis=1)
tabela_contingencia["% Desassistido"] = (
    tabela_contingencia["Desassistido"] / tabela_contingencia["Total"] * 100
).round(2)

print("\nTabela de contingência (cotistas, ponderada por TOTAL_ALUNOS):")
display(tabela_contingencia)

chi2, p_valor, gl, esperado = chi2_contingency(
    tabela_contingencia[["Assistido", "Desassistido"]].values
)

print("\n" + "=" * 60)
print("RESPOSTA — PERGUNTA 2 (Teste Qui-Quadrado de Independência)")
print("=" * 60)
print(f"Estatística Qui-Quadrado: {chi2:.4f}")
print(f"Graus de liberdade: {gl}")
print(f"P-valor: {p_valor:.6f}")
print(f"Significativo a 5%  (p < 0,05):  {'SIM' if p_valor < 0.05 else 'NÃO'}")
print(f"Significativo a 0,5% (p < 0,005): {'SIM' if p_valor < 0.005 else 'NÃO'}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(
    data=tabela_contingencia.reset_index(),
    x="PERIODO",
    y="% Desassistido",
    hue="PERIODO",
    palette=["#3182bd", "#e6550d"],
    legend=False,
    ax=ax,
)
ax.set_title("Taxa de Demanda Não Atendida entre Cotistas — Diurno vs. Noturno", fontweight="bold")
ax.set_ylabel("% de cotistas desassistidos")
ax.set_xlabel("")
for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:.2f}%",
        (p.get_x() + p.get_width() / 2.0, height),
        ha="center",
        va="bottom",
        fontweight="bold",
        xytext=(0, 3),
        textcoords="offset points",
    )
sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()

## Pergunta 3 — Quais os 10 cursos com maior volume absoluto de demanda não atendida?

Diferente da Pergunta 2 (taxa relativa), aqui o critério é **volume absoluto** (`TOTAL_IDPNA`) — relevante para priorização orçamentária, já que cursos pequenos podem ter taxa alta mas poucos alunos afetados, e vice-versa.

In [ ]:
top10_cursos = (
    df_fato.groupby(["CO_CURSO", "CURSO"])
    .agg(
        TOTAL_ALUNOS=("TOTAL_ALUNOS", "sum"),
        TOTAL_IDPNA=("TOTAL_IDPNA", "sum"),
    )
    .reset_index()
)
top10_cursos["% DO CURSO EM IDPNA"] = (
    top10_cursos["TOTAL_IDPNA"] / top10_cursos["TOTAL_ALUNOS"] * 100
).round(2)
top10_cursos = top10_cursos.sort_values("TOTAL_IDPNA", ascending=False).head(10)

print("=" * 60)
print("RESPOSTA — PERGUNTA 3")
print("=" * 60)
display(top10_cursos[["CURSO", "TOTAL_ALUNOS", "TOTAL_IDPNA", "% DO CURSO EM IDPNA"]])

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top10_cursos.sort_values("TOTAL_IDPNA"),
    x="TOTAL_IDPNA",
    y="CURSO",
    hue="CURSO",
    palette="rocket",
    legend=False,
    ax=ax,
)
ax.set_title("Top 10 Cursos — Volume Absoluto de Demanda Não Atendida (IDPNA)", fontweight="bold")
ax.set_xlabel("Total de alunos em situação de IDPNA")
ax.set_ylabel("")
sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()

## Pergunta 4 — Perfil demográfico dos assistidos vs. corpo discente total

Comparamos a distribuição de **raça/cor** e **sexo** entre (a) o corpo discente total do Campus I e (b) o subconjunto de estudantes que efetivamente recebem apoio social (`RECEBE_AUXILIO = 1`). Diferenças positivas indicam grupos **sobrerrepresentados** entre os assistidos em relação à sua presença geral no campus; diferenças negativas indicam **sub-representação**.

In [ ]:
def distribuicao_percentual(df, coluna, filtro=None):
    dados = df if filtro is None else df[filtro]
    dist = dados.groupby(coluna)["TOTAL_ALUNOS"].sum()
    return (dist / dist.sum() * 100).round(2)


filtro_assistidos = df_fato["RECEBE_AUXILIO"] == 1

# --- Raça/Cor ---
dist_raca_total = distribuicao_percentual(df_fato, "RACA_DESCRICAO")
dist_raca_assistidos = distribuicao_percentual(df_fato, "RACA_DESCRICAO", filtro_assistidos)

comparativo_raca = pd.DataFrame(
    {
        "% Corpo Discente Total": dist_raca_total,
        "% Entre Assistidos": dist_raca_assistidos,
    }
).fillna(0)
comparativo_raca["Diferença (p.p.)"] = (
    comparativo_raca["% Entre Assistidos"] - comparativo_raca["% Corpo Discente Total"]
).round(2)
comparativo_raca = comparativo_raca.sort_values("Diferença (p.p.)", ascending=False)

# --- Sexo ---
dist_sexo_total = distribuicao_percentual(df_fato, "SEXO_DESCRICAO")
dist_sexo_assistidos = distribuicao_percentual(df_fato, "SEXO_DESCRICAO", filtro_assistidos)

comparativo_sexo = pd.DataFrame(
    {
        "% Corpo Discente Total": dist_sexo_total,
        "% Entre Assistidos": dist_sexo_assistidos,
    }
).fillna(0)
comparativo_sexo["Diferença (p.p.)"] = (
    comparativo_sexo["% Entre Assistidos"] - comparativo_sexo["% Corpo Discente Total"]
).round(2)

print("=" * 60)
print("RESPOSTA — PERGUNTA 4")
print("=" * 60)
print("\n--- Raça/Cor ---")
display(comparativo_raca)
print("\n--- Sexo ---")
display(comparativo_sexo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparativo_raca[["% Corpo Discente Total", "% Entre Assistidos"]].plot(
    kind="bar", ax=axes[0], color=["#9ecae1", "#e6550d"]
)
axes[0].set_title("Raça/Cor: Corpo Discente vs. Assistidos", fontweight="bold")
axes[0].set_ylabel("% do grupo")
axes[0].tick_params(axis="x", rotation=30)

comparativo_sexo[["% Corpo Discente Total", "% Entre Assistidos"]].plot(
    kind="bar", ax=axes[1], color=["#9ecae1", "#e6550d"]
)
axes[1].set_title("Sexo: Corpo Discente vs. Assistidos", fontweight="bold")
axes[1].set_ylabel("% do grupo")
axes[1].tick_params(axis="x", rotation=0)

sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()

## Síntese para a PRAPE-UFPB

Os resultados desta análise evidenciam que o principal desafio da assistência estudantil no Campus I da UFPB não está na identificação dos estudantes potencialmente elegíveis, mas na diferença entre a demanda potencial e a capacidade atual de atendimento. O Indicador de Demanda Potencial Não Atendida (IDPNA) foi desenvolvido justamente para mensurar esse hiato de cobertura entre estudantes ingressantes por reserva de vagas e o efetivo recebimento de apoio social, constituindo um indicador de gestão para apoiar o planejamento institucional.

1. **Cobertura da assistência estudantil:** entre os **13.303 estudantes ingressantes por reserva de vagas**, apenas **1.573 (11,82%)** recebem algum tipo de apoio social institucional. Consequentemente, **11.730 estudantes (88,18%)** encontram-se classificados como IDPNA. Em termos práticos, isso significa que aproximadamente **nove em cada dez estudantes cotistas** permanecem sem registro de assistência estudantil na base analisada, evidenciando um hiato expressivo entre a demanda potencial e a cobertura observada.

2. **Diferenças entre os turnos:** a demanda potencial não atendida distribui-se de forma desigual entre os turnos. Nos cursos diurnos, **84,47%** dos estudantes cotistas permanecem sem assistência, enquanto no turno noturno esse percentual alcança **94,89%**. O teste do Qui-Quadrado confirmou associação estatisticamente significativa entre o turno e a condição de receber assistência (**χ² = 268,09; p < 0,001**). Embora esse resultado não estabeleça relação de causa e efeito, ele indica que o turno constitui um fator relevante para o monitoramento da política de permanência.

3. **Concentração da demanda:** a distribuição da IDPNA também não é homogênea entre os cursos. Apenas **10 dos 94 cursos de graduação** concentram **3.287 estudantes**, correspondendo a aproximadamente **28% de toda a demanda potencial não atendida** do Campus I. Essa concentração sugere que estratégias focalizadas podem produzir ganhos mais expressivos de cobertura do que intervenções distribuídas uniformemente entre todos os cursos.

4. **Perfil dos beneficiários:** a comparação entre a composição do conjunto de estudantes cotistas e a dos estudantes assistidos mostra diferenças relevantes na distribuição dos benefícios. Os estudantes **pardos** apresentam participação superior à esperada entre os beneficiários (**+9,53 pontos percentuais**), enquanto os grupos **"não informado" (-9,21 p.p.)** e **pretos (-2,11 p.p.)** aparecem sub-representados. Esses resultados não permitem concluir, isoladamente, que exista desigualdade na política de assistência, pois a concessão dos benefícios depende de critérios socioeconômicos que não estão disponíveis na base utilizada. Ainda assim, constituem indicadores relevantes para o monitoramento contínuo da equidade na distribuição dos auxílios.

### Considerações finais

O estudo demonstra que a IDPNA pode ser utilizada como um indicador estratégico de apoio à gestão da assistência estudantil. Ao identificar **onde** a demanda potencial se concentra (cursos), **em quais grupos** ela é mais elevada (turnos e perfis sociodemográficos) e **qual é a magnitude do hiato de cobertura**, o indicador oferece subsídios objetivos para o acompanhamento da política institucional, a definição de prioridades e a avaliação da evolução da cobertura ao longo do tempo. Por se tratar de um indicador de demanda potencial, seus resultados devem ser interpretados em conjunto com informações socioeconômicas e critérios institucionais de concessão dos benefícios.